<a href="https://colab.research.google.com/github/pathilink/adyen_payment_optimization_case/blob/main/notebooks/03_exploratory_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# <font color='#0ABF56'> Optimisation Data Analyst Case Study </font>

## <font color='#0ABF56'> 03 - Exploratory Analysis </font>

# Libraries

In [1]:
import pandas as pd
import numpy as np
import datetime
import seaborn as sns
from matplotlib import pyplot as plt

# Data

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
df = pd.read_csv(
    '/content/drive/MyDrive/test/adyen/data/processed/adyen_transactions_clean.csv',
    dtype={'bin': str},
    parse_dates=['creation_date']
)

df_ = df.copy()

df_.head()

,psp_reference,bin,scheme,issuername,shopper_interaction,avs_data_supplied,cvc_data_supplied,amount,raw_acquirer_response,creation_date,authorization
0,1,400178,visa,BANCO DO BRASIL S.A.,Ecommerce,False,False,1.00,05 : Do not honor / A201 : 3D Secure Mandated,2019-06-01 00:19:00,False
1,2,486348,visa,FIRST ATLANTIC BANK LIMITED,Ecommerce,False,False,6.48,00 : Approved or completed successfully,2019-06-01 00:22:00,True
2,3,482481,visa,ITAU UNIBANCO S.A.,Ecommerce,False,True,4.00,06 : Error,2019-06-01 00:46:00,False
3,4,439267,visa,CAIXA ECONOMICA FEDERAL,Ecommerce,False,True,2.76,05 : Do not honor / A201 : 3D Secure Mandated,2019-06-01 01:02:00,False
4,5,489347,visa,VTB BANK PJSC,Ecommerce,False,True,97.00,00 : Approved or completed successfully,2019-06-01 01:30:00,True


In [4]:
df_.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 933387 entries, 0 to 933386
Data columns (total 11 columns):
 #   Column                 Non-Null Count   Dtype         
---  ------                 --------------   -----         
 0   psp_reference          933387 non-null  int64         
 1   bin                    933376 non-null  object        
 2   scheme                 933387 non-null  object        
 3   issuername             932580 non-null  object        
 4   shopper_interaction    933387 non-null  object        
 5   avs_data_supplied      933387 non-null  bool          
 6   cvc_data_supplied      933387 non-null  bool          
 7   amount                 933387 non-null  float64       
 8   raw_acquirer_response  933387 non-null  object        
 9   creation_date          933387 non-null  datetime64[ns]
 10  authorization          933387 non-null  bool          
dtypes: bool(3), datetime64[ns](1), float64(1), int64(1), object(5)
memory usage: 59.6+ MB


# EDA

## Metrics

In [5]:
# min and max dates
min_date = df['creation_date'].min()
max_date = df['creation_date'].max()

print(f'Minimum date: {min_date}')
print(f'Maximum date: {max_date}')

Minimum date: 2019-06-01 00:00:00
Maximum date: 2019-06-30 23:59:00


In [6]:
print(f'Total transactions: {df_['psp_reference'].nunique()}')

Total transactions: 933387


In [7]:
print(f'Authorizantion rate: { \
    round(
    df_.query('authorization == True').shape[0] / df_['psp_reference'].nunique()
    * 100, 2
    ) \
}%')

Authorizantion rate: 79.3%


## Scheme performance

* Scheme (Visa, MC) proportion
* Which scheme authorizes more?

In [8]:
scheme_summary = (
    df_.groupby('scheme')
    .agg(
        # percentage of transactions for each scheme out of the total
        transaction_share=('scheme', lambda x: (len(x) / len(df_)) * 100),
        # authorization rate (average of True/False multiplied by 100)
        authorization_rate=('authorization', lambda x: x.mean() * 100)
    )
    .round(2)
    .reset_index()
    .sort_values(by='transaction_share', ascending=False)
)

scheme_summary

,scheme,transaction_share,authorization_rate
0,mc,66.29,80.56
1,visa,33.71,76.84


* Overall Average (79.30%): This is the break-even point.

* Mastercard (80.56%): This is 1.26 percentage points above the overall industry average.

* Visa (76.84%): This is 2.46 percentage points below the overall industry average.

## Issuer’s authorisation

The authorisation rate depends on the issuer. Some banks may have different rules for authorising a transaction.

* Which issuing banks have the lowest approval rate?

In [9]:
# flag for non-null issuer name
df_['issuer_known'] = df_['issuername'].notna()

df_[['issuername', 'issuer_known']].sample(5)

,issuername,issuer_known
398770,BANCO BRADESCO S.A.,True
911239,OMNI S.A. CREDITO FINANCIAMENTO E INVESTIMENTO,True
606116,ITAU UNIBANCO S.A.,True
216361,ITAU UNIBANCO S.A.,True
620690,ITAU UNIBANCO S.A.,True


In [10]:
# same baseline
print(f'Authorizantion rate with known issuer: { \
    round(
    df_
    .query('issuer_known == True')
    .query('authorization == True').shape[0] / df_['psp_reference'].nunique()
    * 100, 2
    ) \
}%')

Authorizantion rate with known issuer: 79.29%


In [11]:
issuer_analysis = (
    df_.query('issuer_known == True')
    .groupby('issuername')
    .agg(
        transactions=('psp_reference', 'count'), # PSP (Payment Service Provider) == Adyen
        auth_rate=('authorization', 'mean')
    )
    .reset_index()
)

issuer_analysis['auth_rate'] = (
    issuer_analysis['auth_rate'] * 100
).round(2)

issuer_analysis = issuer_analysis.query('transactions >= 1000') # at least

issuer_analysis.sort_values('auth_rate').head(20)

,issuername,transactions,auth_rate
251,BPP INSTITUICAO DE PAGAMENTO S.A.,1089,32.97
642,SUPER PAGAMENTOS E ADMINISTRAC,4186,34.52
69,BANCO CETELEM S.A,4429,47.98
40,BANCO AGIBANK S.A.,1314,48.55
59,BANCO BMG S/A,1864,48.55
423,HUB PAGAMENTOS S.A.,6818,52.08
6,ACESSO SOLUCOES DE PAGAMENTO S,3554,54.22
141,BANCO INTER S.A.,7654,57.93
544,PAGSEGURO INTERNET LTDA,6946,58.02
63,BANCO BRADESCARD S.A.,17462,59.54


Several specific Brazilian issuers have authorisation rates that are well below average (79%).

* Most of them are fintech companies and microcredit providers.

| Hypothesis | Explanation |
| :- | :- |
| Greater susceptibility to fraud | issuers are more likely to decline transactions as a precaution |
| Riskier credit profile | higher rate of insufficient funds |
| Higher volume of risky e-commerce | issuers tighten rules |
| Shorter anti-fraud maturity | more “Do Not Honour” |
| Problems with recurring payments | poorer account authentication at these banks |

<br>

Issues relating to authorisation appear to be concentrated among a small number of issuers, suggesting opportunities for optimisation within a specific segment.

### Check issuers

In [12]:
top10_issuers_non_authoritative = issuer_analysis.sort_values('auth_rate', ascending=True).head(10)['issuername'].tolist()

top10_issuers_non_authoritative

['BPP INSTITUICAO DE PAGAMENTO S.A.',
 'SUPER PAGAMENTOS E ADMINISTRAC',
 'BANCO CETELEM S.A',
 'BANCO AGIBANK S.A.',
 'BANCO BMG S/A',
 'HUB PAGAMENTOS S.A.',
 'ACESSO SOLUCOES DE PAGAMENTO S',
 'BANCO INTER S.A.',
 'PAGSEGURO INTERNET LTDA',
 'BANCO BRADESCARD S.A.']

**issuer × raw_acquirer_response**

In [13]:
# issuer_response = (
#     df_
#     .query('issuername in @top10_issuers_non_authoritative')
#     .groupby(
#         ['issuername', 'raw_acquirer_response']
#     )
#     .size()
#     .reset_index(name='transactions')
#     .query('transactions >= 1000')
#     .sort_values('transactions', ascending=False)
# )

# issuer_response.head(10)

In [14]:
# total number of transactions for the issuer with the specific raw_acquirer_response
grouped = (
    df_
    .query('issuername in @top10_issuers_non_authoritative')
    .groupby(['issuername', 'raw_acquirer_response'])
    .size()
    .reset_index(name='transactions')
)

# calculate the rate
# transform(“sum”) will sum the “transactions” column for each 'issuername'
grouped['transactions_rate'] = grouped['transactions'] / grouped.groupby('issuername')['transactions'].transform('sum')

# applies the minimum volume filter and sorts the results
final_result = (
    grouped
    .query('transactions >= 1000') # at least
    .sort_values(['issuername', 'transactions_rate'], ascending=[True, False]) # Ordena por emissor e depois pelas maiores taxas
)

final_result

,issuername,raw_acquirer_response,transactions,transactions_rate
0,ACESSO SOLUCOES DE PAGAMENTO S,00 : Approved or completed successfully,1878,0.528419
4,ACESSO SOLUCOES DE PAGAMENTO S,51 : Insufficient funds/over credit limit,1434,0.403489
29,BANCO BRADESCARD S.A.,00 : Approved or completed successfully,10276,0.588478
31,BANCO BRADESCARD S.A.,05 : Do not honor,5050,0.289199
42,BANCO CETELEM S.A,00 : Approved or completed successfully,1996,0.450666
51,BANCO CETELEM S.A,77 : Invalid/nonexistent ‚ÄúFrom Account‚Äù sp...,1793,0.404832
53,BANCO INTER S.A.,00 : Approved or completed successfully,4356,0.569114
54,BANCO INTER S.A.,05 : Do not honor,2215,0.289391
72,HUB PAGAMENTOS S.A.,00 : Approved or completed successfully,3551,0.520827
75,HUB PAGAMENTOS S.A.,51 : Insufficient funds/over credit limit,2789,0.409064


**The main rejections among the top 10 issuers with the most cancelled transactions**

| Reason | Description |
|:-|:-|
| 00 : Approved or completed successfully | The transaction was canceled after being initially approved by the issuer. <br>This can be due to various reasons, for example, if the shopper returns goods after purchase. |
| 51 : Insufficient funds/over credit limit | Insufficient funds in the cardholder's account. <br>The shopper can try again after adding funds to their bank account, or use another payment method. |
| 05 : Do not honor| 	The card issuer requests to retain the card. This can be due to a suspected counterfeit or stolen card. <br>This reason is used in an ecommerce environment although it originates from an in-person payments environment. |
| 77 : Invalid/nonexistent "from" account specified | Decline |
| 57 : Transaction not permitted to issuer/cardholder | Decline |





**issuer × shopper_interaction**

In [15]:
(
    df_
    .query('issuername in @top10_issuers_non_authoritative')
    .groupby(
    ['issuername', 'shopper_interaction']
    )['authorization'].mean()
    .reset_index()
)

,issuername,shopper_interaction,authorization
0,ACESSO SOLUCOES DE PAGAMENTO S,ContAuth,0.597538
1,ACESSO SOLUCOES DE PAGAMENTO S,Ecommerce,0.495594
2,BANCO AGIBANK S.A.,ContAuth,0.588785
3,BANCO AGIBANK S.A.,Ecommerce,0.452165
4,BANCO BMG S/A,ContAuth,0.512528
5,BANCO BMG S/A,Ecommerce,0.477193
6,BANCO BRADESCARD S.A.,ContAuth,0.623634
7,BANCO BRADESCARD S.A.,Ecommerce,0.585922
8,BANCO CETELEM S.A,ContAuth,0.285609
9,BANCO CETELEM S.A,Ecommerce,0.787967


* Market Trend: ContAuth (recurring) transactions tend to have higher approval rates than traditional e-commerce transactions (e.g. Banco Inter approves 74.5% vs. 54%), as they are viewed as more secure by banks.

* The Exception (Banco Cetelem): Takes the opposite approach with divergent behaviour — it has the worst approval rate for ContAuth (28.5%) and the best for E-commerce (78.7%).

* Below Average: BPP and Super Pagamentos record the worst overall performance, hovering between 32% and 38% approval rates in both categories.

**issuer × cvc_data_supplied**

In [16]:
(
    df_
    .query('issuername in @top10_issuers_non_authoritative')
    .groupby(
      ['issuername', 'cvc_data_supplied']
      )['authorization'].mean()
    .reset_index()
)

,issuername,cvc_data_supplied,authorization
0,ACESSO SOLUCOES DE PAGAMENTO S,False,0.552275
1,ACESSO SOLUCOES DE PAGAMENTO S,True,0.518130
2,BANCO AGIBANK S.A.,False,0.523490
3,BANCO AGIBANK S.A.,True,0.435852
4,BANCO BMG S/A,False,0.486637
5,BANCO BMG S/A,True,0.484472
6,BANCO BRADESCARD S.A.,False,0.603392
7,BANCO BRADESCARD S.A.,True,0.587101
8,BANCO CETELEM S.A,False,0.341165
9,BANCO CETELEM S.A,True,0.805598


Submitting the CVC does not guarantee a higher approval rate: the impact varies depending on the issuing bank.

* Banco Inter performs significantly better without the CVC (73.3%)
* Banco Cetelem requires the CVC to achieve an acceptable approval rate (jumping from 34.1% to 80.5%).

**issuer × amount**

In [17]:
df_['amount'].describe()

,amount
count,933387.000000
mean,72.038756
std,265.074149
min,0.000000
25%,4.160000
50%,19.550000
75%,52.100000
max,22192.000000


In [18]:
bins = [-np.inf, 4.16, 19.55, 52.10, np.inf]

labels = ["Until 4.16", "4.16 to 19.55", "19.55 to 52.10", "above 52.10"]

# range column
df_["amount_range"] = pd.cut(df_["amount"], bins=bins, labels=labels)

In [19]:
(
    df_
    .query('issuername in @top10_issuers_non_authoritative')
    .groupby(
    ['issuername', 'amount_range'], observed=True
    )['authorization'].mean()
    .reset_index()
)

,issuername,amount_range,authorization
0,ACESSO SOLUCOES DE PAGAMENTO S,Until 4.16,0.542056
1,ACESSO SOLUCOES DE PAGAMENTO S,4.16 to 19.55,0.535789
2,ACESSO SOLUCOES DE PAGAMENTO S,19.55 to 52.10,0.545783
3,ACESSO SOLUCOES DE PAGAMENTO S,above 52.10,0.547739
4,BANCO AGIBANK S.A.,Until 4.16,0.453427
5,BANCO AGIBANK S.A.,4.16 to 19.55,0.539474
6,BANCO AGIBANK S.A.,19.55 to 52.10,0.489362
7,BANCO AGIBANK S.A.,above 52.10,0.461538
8,BANCO BMG S/A,Until 4.16,0.478070
9,BANCO BMG S/A,4.16 to 19.55,0.506787


Does the ticket value affect the likelihood of a transaction being approved?

Yes, the ticket value has a significant impact on approval rates, and banks handle this differently.

* Larger tickets result in fewer approvals
* Very low-value transactions tend to exhibit extreme behaviour: they are either approved very frequently because they pose a low financial risk, or they are declined very often because they are associated with card testing by scammers (carding) - excepts Banco Cetelem

**issuer × amount_range x shopper_interaction**

In [20]:
pivot_ratio = (
    df_
    .query('issuername in @top10_issuers_non_authoritative')
    .pivot_table(
        index='amount_range',            # rows
        columns='shopper_interaction',   # columns
        values='authorization',          # to calculate
        aggfunc='mean',                  # Metric (average of the Boolean column = pass rate)
        observed=True                    # avoid warning
    )
)

# Multiplicando por 100 e arredondando para ver como porcentagem (%)
pivot_ratio_pct = (pivot_ratio * 100).round(2)

pivot_ratio_pct.style.background_gradient(cmap='RdYlGn', axis=None)

shopper_interaction,ContAuth,Ecommerce
amount_range,,
Until 4.16,64.400000,54.010000
4.16 to 19.55,51.230000,53.200000
19.55 to 52.10,51.510000,58.050000
above 52.10,57.920000,50.900000


* The approval rate for high-value e-commerce transactions (above 52.10) is 50.90%, which is the lowest rate in the entire e-commerce column.

* Issuers tend to approve more smaller amounts when they are recurring.

**proportion of refusals**

In [21]:
# filter
df_declined = df_[df_['authorization'] == False]

# volume de recusas por banco e calculando a proporção sobre o total
share_decline = (
    df_declined['issuername']
    .value_counts(normalize=True) # calculate each share (0 to 1)
    .reset_index()
)

share_decline.columns = ['issuername', 'proportion_total_refusals']
share_decline['proportion_total_refusals (%)'] = (share_decline['proportion_total_refusals'] * 100).round(2)

share_decline.drop(
        columns=['proportion_total_refusals'],
        inplace=True,
        errors='ignore'
    )

df_top10_refusals = share_decline.query('issuername in @top10_issuers_non_authoritative')
df_top10_refusals

,issuername,proportion_total_refusals (%)
7,BANCO BRADESCARD S.A.,3.67
9,HUB PAGAMENTOS S.A.,1.70
10,BANCO INTER S.A.,1.67
12,PAGSEGURO INTERNET LTDA,1.51
13,SUPER PAGAMENTOS E ADMINISTRAC,1.42
16,BANCO CETELEM S.A,1.20
18,ACESSO SOLUCOES DE PAGAMENTO S,0.85
23,BANCO BMG S/A,0.50
27,BPP INSTITUICAO DE PAGAMENTO S.A.,0.38
28,BANCO AGIBANK S.A.,0.35


In [22]:
total_top10 = df_top10_refusals['proportion_total_refusals (%)'].sum()

print(f"\nTotal accumulated by the Top 10: {total_top10:.2f}%")


Total accumulated by the Top 10: 13.25%


The group of 10 issuers with the lowest transaction approval rates accounts for just 13.25% of total rejections. Therefore, even if a solution were devised to approve 100% of their transactions, the maximum potential for revenue recovery would be low. The majority of lost transactions (86.75%) are spread across the rest of the market.

## Conclusion

Although the 10 issuers operate at below-average efficiency (79.30%), they account for only 13.25% of rejections.

Efforts should be focused on the emitters that account for the remaining 86.75% of rejections, where small improvements in efficiency will yield a greater absolute financial return.

## Download

In [23]:
output_path = '/content/drive/MyDrive/test/adyen/data/processed/adyen_transactions_analysis.csv'

df_.to_csv(output_path, index=False)